In [3]:
# Ucitavanje svih relevantnih tabela iz preprocesiranih skupova podataka

players = pd.read_csv("../data/processed/players_clean.csv")
appearances = pd.read_csv("../data/processed/appearances_clean.csv")
valuations = pd.read_csv("../data/processed/player_valuations_clean.csv")
transfers = pd.read_csv("../data/processed/transfers_clean.csv")
lineups = pd.read_csv("../data/processed/game_lineups_clean.csv")
events = pd.read_csv("../data/processed/game_events_clean.csv")
clubs = pd.read_csv("../data/processed/clubs_clean.csv")
national_teams = pd.read_csv("../data/processed/national_teams_clean.csv")

In [6]:
# Zelimo da napravimo jednu tabelu koja ce objediniti sve bitne podatke vezane za jednog igraca. Na ovaj nacin cemo lakse moci
# da ih klasterujemo i uocavamo pravilnosti

# Biramo tabelu players kao pocetnu jer ona sadrzi najvise podataka o igracima
player_features = players.copy() 

# Iz nje biramo podatke koji su relevantni za dalji rad
base_columns = [
    "player_id",
    "age",
    "height_in_cm",
    "international_caps",
    "international_goals",
    "market_value_in_eur",
    "highest_market_value_in_eur",
    "position",
    "sub_position",
    "foot",
    "country_of_birth",
    "country_of_citizenship",
    "current_club_id",
    "current_club_domestic_competition_id"
]

player_features = players[base_columns].copy()

In [12]:
player_features.shape

(44905, 14)

In [15]:
# Agregiramo informacije iz tabele appearances po player_id atributu

appearance_features = appearances.groupby("player_id").agg(
    matches_played=("appearance_id", "count"),
    total_minutes=("minutes_played", "sum"),
    avg_minutes=("minutes_played", "mean"),
    max_minutes=("minutes_played", "max"),
    minutes_std=("minutes_played", "std"),

    total_goals=("goals", "sum"),
    total_assists=("assists", "sum"),
    total_yellow_cards=("yellow_cards", "sum"),
    total_red_cards=("red_cards", "sum")
).reset_index()

appearance_features["goals_per_match"] = appearance_features["total_goals"] / appearance_features["matches_played"]
appearance_features["assists_per_match"] = appearance_features["total_assists"] / appearance_features["matches_played"]
appearance_features["yellow_per_match"] = appearance_features["total_yellow_cards"] / appearance_features["matches_played"]
appearance_features["red_per_match"] = appearance_features["total_red_cards"] / appearance_features["matches_played"]

print(appearance_features.shape)
appearance_features.head()

(28152, 14)


,player_id,matches_played,total_minutes,avg_minutes,max_minutes,minutes_std,total_goals,total_assists,total_yellow_cards,total_red_cards,goals_per_match,assists_per_match,yellow_per_match,red_per_match
0,10,136,8808,64.764706,90,29.232462,48,25,19,0,0.352941,0.183824,0.139706,0.000000
1,26,152,13508,88.868421,120,11.379477,0,0,4,2,0.000000,0.000000,0.026316,0.013158
2,65,122,8788,72.032787,90,26.330680,38,13,11,1,0.311475,0.106557,0.090164,0.008197
3,77,4,307,76.750000,120,41.451779,0,0,0,0,0.000000,0.000000,0.000000,0.000000
4,80,12,1080,90.000000,90,0.000000,0,0,0,0,0.000000,0.000000,0.000000,0.000000


In [16]:
# Spajamo izdvojene atribute sa players_features i to sa leve strane jer mozemo da vidimo da je od nasih 44905 igraca samo njih
# 28152 zabelezilo nastupe. Nedostajuce vrednosti koje se ticu nastupa cemo popuniti nulama jer to ne znaci da informacije nedostaju vec da jednostavno
# ne postoje, te je i ocekivano da minuti, golovi, asistencije, kao i sve ostale informacije budu 0 za te igrace.

player_features = player_features.merge(
    appearance_features,
    on="player_id",
    how="left"
)

print(player_features.shape)

appearance_cols = [col for col in appearance_features.columns if col != "player_id"]

player_features[appearance_cols] = player_features[appearance_cols].fillna(0)

(44905, 27)


In [21]:
# Istu pricu ponavljamo i za ostale tabele koje su nam znacajne. Prva na redu je valuations tabela.
import numpy as np

valuations = valuations.sort_values(["player_id", "date"])

valuation_features = valuations.groupby("player_id").agg(
    valuation_count=("market_value_in_eur", "count"),
    avg_valuation=("market_value_in_eur", "mean"),
    min_valuation=("market_value_in_eur", "min"),
    max_valuation=("market_value_in_eur", "max"),
    std_valuation=("market_value_in_eur", "std"),
    first_valuation=("market_value_in_eur", "first"),
    latest_valuation=("market_value_in_eur", "last"),
).reset_index()

valuation_features["valuation_growth"] = (
    valuation_features["latest_valuation"] -
    valuation_features["first_valuation"]
)

valuation_features["valuation_growth_pct"] = np.where(
    valuation_features["first_valuation"] > 0,
    valuation_features["valuation_growth"] / valuation_features["first_valuation"],
    0
)

valuation_features["std_valuation"] = valuation_features["std_valuation"].fillna(0)

valuation_features.shape

(39361, 10)

In [22]:
# Kao i za nastupe, i za valuacije cemo nedostajuce informacije popuniti nulama. Ovo je u redu jer smo i na pocetku ustanovili da cemo transfere sa
# nedostajucim naknadama tretirati kao da su bili besplatni.

player_features = player_features.merge(
    valuation_features,
    on="player_id",
    how="left"
)

valuation_cols = [col for col in valuation_features.columns if col != "player_id"]
player_features[valuation_cols] = player_features[valuation_cols].fillna(0)

player_features.shape

(44905, 54)

In [23]:
# Transferi

transfers["transfer_fee"] = transfers["transfer_fee"].fillna(0)
transfers["market_value_in_eur"] = transfers["market_value_in_eur"].fillna(0)

transfers["is_paid_transfer"] = (transfers["transfer_fee"] > 0).astype(int)

transfer_features = transfers.groupby("player_id").agg(
    transfer_count=("transfer_date", "count"),
    total_transfer_fee=("transfer_fee", "sum"),
    avg_transfer_fee=("transfer_fee", "mean"),
    max_transfer_fee=("transfer_fee", "max"),
    paid_transfer_count=("is_paid_transfer", "sum"),
    avg_transfer_market_value=("market_value_in_eur", "mean"),
    max_transfer_market_value=("market_value_in_eur", "max"),
    clubs_from_count=("from_club_id", "nunique"),
    clubs_to_count=("to_club_id", "nunique"),
).reset_index()

transfer_features["clubs_changed"] = (
    transfer_features["clubs_from_count"] +
    transfer_features["clubs_to_count"]
)

transfer_features.shape

(21023, 11)

In [24]:
player_features = player_features.merge(
    transfer_features,
    on="player_id",
    how="left"
)

transfer_cols = [col for col in transfer_features.columns if col != "player_id"]
player_features[transfer_cols] = player_features[transfer_cols].fillna(0)

player_features.shape

(44905, 64)

In [25]:
# Sledeca tabela su game_lineups. Iz nje cemo izvuci informacije o tome koliko puta je igrac startovao, koliko puta je bio izmena, koliko puta kapiten,
# kao i najbitnije, na kojim pozicijama je igrao

lineups["is_starter"] = (lineups["type"] == "starting_lineup").astype(int)
lineups["is_substitute"] = (lineups["type"] == "substitutes").astype(int)

lineup_features = lineups.groupby("player_id").agg(
    lineup_count=("game_lineups_id", "count"),
    starter_count=("is_starter", "sum"),
    substitute_count=("is_substitute", "sum"),
    captain_count=("team_captain", "sum"),
).reset_index()

lineup_features["starter_ratio"] = (
    lineup_features["starter_count"] / lineup_features["lineup_count"]
)

lineup_features["captain_ratio"] = (
    lineup_features["captain_count"] / lineup_features["lineup_count"]
)

lineup_features.shape

(106482, 7)

In [26]:
lineups["player_id"].nunique()

106482

In [29]:
# Mozemo videti da u lineups postoji vise igraca nego sto imamo u tabeli players. Iz tog razloga cemo iz ove tabele da izbacimo sve igrace o kojima
# nemamo dodatne informacije, a zatim kao i za prethodne tabele spojiti sa player_features

valid_player_ids = set(players["player_id"])

lineups = lineups[lineups["player_id"].isin(valid_player_ids)].copy()

print(lineups["player_id"].nunique())
print(lineups.shape)

39764
(2502126, 12)


In [30]:
lineups["is_starter"] = (lineups["type"] == "starting_lineup").astype(int)
lineups["is_substitute"] = (lineups["type"] == "substitutes").astype(int)

lineup_features = lineups.groupby("player_id").agg(
    lineup_count=("game_lineups_id", "count"),
    starter_count=("is_starter", "sum"),
    substitute_count=("is_substitute", "sum"),
    captain_count=("team_captain", "sum"),
).reset_index()

lineup_features["starter_ratio"] = lineup_features["starter_count"] / lineup_features["lineup_count"]
lineup_features["captain_ratio"] = lineup_features["captain_count"] / lineup_features["lineup_count"]

lineup_features.shape

(39764, 7)

In [31]:
player_features = player_features.merge(
    lineup_features,
    on="player_id",
    how="left"
)

lineup_cols = [col for col in lineup_features.columns if col != "player_id"]
player_features[lineup_cols] = player_features[lineup_cols].fillna(0)

player_features.shape

(44905, 70)

In [34]:
# sada izvlacimo pozicije

lineups["position_clean"] = (
    lineups["position"]
    .fillna("Unknown")
    .str.lower()
    .str.strip()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

position_counts = (
    lineups
    .groupby(["player_id", "position_clean"])
    .size()
    .unstack(fill_value=0)
)

position_counts.columns = [
    f"games_as_{col}" for col in position_counts.columns
]

position_counts = position_counts.reset_index()
position_counts.shape

(39764, 18)

In [35]:
# Procenat odigranih utakmica na odredjenoj poziciji

position_cols = [col for col in position_counts.columns if col != "player_id"]

position_percentages = position_counts[["player_id"]].copy()

total_position_games = position_counts[position_cols].sum(axis=1)

for col in position_cols:
    pct_col = col.replace("games_as_", "pct_as_")
    position_percentages[pct_col] = position_counts[col] / total_position_games

In [36]:
position_features = position_counts.merge(
    position_percentages,
    on="player_id",
    how="left"
)

player_features = player_features.merge(
    position_features,
    on="player_id",
    how="left"
)

position_feature_cols = [col for col in position_features.columns if col != "player_id"]
player_features[position_feature_cols] = player_features[position_feature_cols].fillna(0)

player_features.shape

(44905, 104)

In [37]:
# game_events

valid_player_ids = set(players["player_id"])
events = events[events["player_id"].isin(valid_player_ids)].copy()

event_type_counts = (
    events
    .groupby(["player_id", "type"])
    .size()
    .unstack(fill_value=0)
)

event_type_counts.columns = [
    "event_" + str(col).lower().replace(" ", "_")
    for col in event_type_counts.columns
]

event_type_counts = event_type_counts.reset_index()

player_features = player_features.merge(
    event_type_counts,
    on="player_id",
    how="left"
)

event_cols = [col for col in event_type_counts.columns if col != "player_id"]
player_features[event_cols] = player_features[event_cols].fillna(0)

player_features.shape

(44905, 108)

In [39]:
# Sada uvodimo atribute vezane za klub u kojem igrac igra. Ovi podaci mogu biti i dobri i losi, buduci da neki od njih mozda nemaju neke vrednosti
# za samog igraca, na primer broj mesta na stadionu ili prosecna starost igraca u klubu. Medjutim, za pocetak cemo da ubacimo bas sve informacije
# koje imaju bilo kakve veze sa igracem, a u sledecem notebook-u cemo videti na koji nacin mozemo eksperimentisati sa razlicitim podskupovima atributa
# i na koji nacin bi to pomoglo klasterovanju.

club_features = clubs[[
    "club_id",
    "squad_size",
    "average_age",
    "foreigners_number",
    "foreigners_percentage",
    "national_team_players",
    "stadium_seats"
]].copy()

club_features = club_features.rename(columns={
    "club_id": "current_club_id",
    "squad_size": "club_squad_size",
    "average_age": "club_average_age",
    "foreigners_number": "club_foreigners_number",
    "foreigners_percentage": "club_foreigners_percentage",
    "national_team_players": "club_national_team_players",
    "stadium_seats": "club_stadium_seats"
})

player_features = player_features.merge(
    club_features,
    on="current_club_id",
    how="left"
)

# Primetimo da ovde nismo popunjavali nedostajuce informacije nulama. Ukoliko igrac nema klub, ne mozemo samo staviti 0. Ovim problemom cemo se baviti
# u narednim delovima rada, kada budemo odlucivali na koji nacin cemo klasifikovati ovakve nedostajuce vrednosti

player_features.shape

(44905, 120)

In [40]:
# Cuvamo ovako spojenu tabelu
player_features.to_csv(
    "../data/processed/player_features_full.csv",
    index=False
)

In [41]:
[col for col in player_features.columns if col.endswith("_x")]

['valuation_count_x',
 'avg_valuation_x',
 'min_valuation_x',
 'max_valuation_x',
 'std_valuation_x',
 'first_valuation_x',
 'latest_valuation_x',
 'valuation_growth_x',
 'valuation_growth_pct_x',
 'club_squad_size_x',
 'club_average_age_x',
 'club_foreigners_number_x',
 'club_foreigners_percentage_x',
 'club_national_team_players_x',
 'club_stadium_seats_x']

In [42]:
[col for col in player_features.columns if col.endswith("_y")]

['valuation_count_y',
 'avg_valuation_y',
 'min_valuation_y',
 'max_valuation_y',
 'std_valuation_y',
 'first_valuation_y',
 'latest_valuation_y',
 'valuation_growth_y',
 'valuation_growth_pct_y',
 'club_squad_size_y',
 'club_average_age_y',
 'club_foreigners_number_y',
 'club_foreigners_percentage_y',
 'club_national_team_players_y',
 'club_stadium_seats_y']

In [43]:
(player_features["valuation_count_x"] ==
 player_features["valuation_count_y"]).all()

np.False_

In [44]:
(player_features["club_squad_size_x"] ==
 player_features["club_squad_size_y"]).all()

np.True_

In [46]:
print(player_features["club_average_age_x"].isna().sum())
player_features["club_average_age_y"].isna().sum()

1813


np.int64(1813)

In [47]:
print(player_features["valuation_count_x"].isna().sum())
print(player_features["valuation_count_y"].isna().sum())

0
5544


In [48]:
cols_to_drop = [col for col in player_features.columns if col.endswith("_y")]

player_features = player_features.drop(columns=cols_to_drop)

rename_dict = {
    col: col[:-2]
    for col in player_features.columns
    if col.endswith("_x")
}

player_features = player_features.rename(columns=rename_dict)

In [49]:
[col for col in player_features.columns if col.endswith("_x") or col.endswith("_y")]

[]

In [50]:
player_features.isna().sum().sort_values(ascending=False).head(20)

international_goals                     30017
international_caps                      30017
highest_market_value_in_eur              5544
market_value_in_eur                      5544
foot                                     4695
country_of_birth                         4585
height_in_cm                             3321
club_foreigners_percentage               2791
club_average_age                         1813
sub_position                              434
country_of_citizenship                    285
age                                        49
player_id                                   0
current_club_domestic_competition_id        0
matches_played                              0
position                                    0
current_club_id                             0
max_minutes                                 0
minutes_std                                 0
total_goals                                 0
dtype: int64

In [51]:
# 1. Reprezentativni nastupi/golovi: NaN -> 0
zero_cols = [
    "international_caps",
    "international_goals",
    "market_value_in_eur",
    "highest_market_value_in_eur"
]

for col in zero_cols:
    if col in player_features.columns:
        player_features[col] = player_features[col].fillna(0)

In [52]:
player_features.isna().sum().sort_values(ascending=False).head(20)

foot                                    4695
country_of_birth                        4585
height_in_cm                            3321
club_foreigners_percentage              2791
club_average_age                        1813
sub_position                             434
country_of_citizenship                   285
age                                       49
international_goals                        0
position                                   0
highest_market_value_in_eur                0
market_value_in_eur                        0
current_club_id                            0
current_club_domestic_competition_id       0
matches_played                             0
total_minutes                              0
avg_minutes                                0
max_minutes                                0
minutes_std                                0
total_goals                                0
dtype: int64

In [53]:
median_cols = [
    "height_in_cm",
    "club_foreigners_percentage",
    "club_average_age",
    "age"
]

for col in median_cols:
    if col in player_features.columns:
        player_features[col] = player_features[col].fillna(
            player_features[col].median()
        )

In [54]:
categorical_missing_cols = [
    "foot",
    "sub_position",
    "country_of_citizenship"
]

for col in categorical_missing_cols:
    if col in player_features.columns:
        player_features[col] = player_features[col].fillna("Unknown")

In [55]:
player_features.isna().sum().sort_values(ascending=False).head(20)

country_of_birth                        4585
age                                        0
height_in_cm                               0
international_caps                         0
international_goals                        0
market_value_in_eur                        0
highest_market_value_in_eur                0
position                                   0
player_id                                  0
sub_position                               0
foot                                       0
country_of_citizenship                     0
current_club_id                            0
current_club_domestic_competition_id       0
matches_played                             0
total_minutes                              0
avg_minutes                                0
max_minutes                                0
minutes_std                                0
total_goals                                0
dtype: int64

In [56]:
player_features.to_csv(
    "../data/processed/player_features_full.csv",
    index=False
)

Ovde je bio problem sto su se neke tabele spojile vise puta i dale _x i _y verzije atributa, pri cemu su _x zapravo bile one koje i treba da ostanu, a _y su bile stari atributi koje smo imputirali. Zbog toga smo uklonili sve _y kolone, preimenovali _x u verzije bez ikakvog nastavka i zatim ponovo sacuvali ceo fajl na mesto starog. Sada je potrebno ponovo ucitati ovaj skup u sledecem notebook-u, a zatim i pokrenuti sve ostale celije.